# COMBINEX Counterfactual Explanation for GMEL

This notebook integrates COMBINEX with the GMEL commuting flow model.

**What it does:**
- Discretises predicted flow into 3 classes (low / medium / high)
- Wraps GMEL as a node-level classifier that COMBINEX can explain
- Runs COMBINEX to find the minimal node-feature changes needed to flip the flow class of an origin census tract

**Key design decision — Linear Probe:**  
COMBINEX requires a differentiable oracle `forward(x, edge_index) → log-softmax`.  
GMEL uses a non-differentiable GBRT predictor. We bridge this by training a logistic regression ("probe") that maps GMEL source embeddings → flow class.  
COMBINEX finds counterfactuals that flip the probe's prediction; those feature changes are then interpretable in terms of the original GMEL predictions.

**Limitation:** The oracle operates on the extracted k-hop subgraph, not the full graph.  
Embeddings computed on the subgraph are an approximation of full-graph embeddings.

## 1. Setup

In [ ]:
import os, sys, random, warnings, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl

# ── Paths ─────────────────────────────────────────────────────────────────────
GMEL_CODE_DIR  = os.path.abspath(".")
COMBINEX_DIR   = os.path.abspath("../CF-models/COMBINEX")

for p in [GMEL_CODE_DIR, COMBINEX_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import utils
from model import MyModel

from omegaconf import OmegaConf
from torch_geometric.data import Data
from src.datasets.dataset import DataInfo
from src.utils.explainer import get_node_explainer
from src.node_level_explainer.utils.utils import build_factual_graph, check_graphs

warnings.filterwarnings("ignore", message="Feature table contains NaN.*")

print("Imports OK")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║              EDIT THESE PARAMETERS                             ║
# ╚══════════════════════════════════════════════════════════════════╝

SRC_NODE  = 25    # origin census tract (node to explain)
DST_NODE  = 19    # destination census tract (fixed for class labelling)

# Flow class boundaries
#   Class 0  "low"    : flow < LOW_THRESH
#   Class 1  "medium" : LOW_THRESH ≤ flow ≤ HIGH_THRESH
#   Class 2  "high"   : flow > HIGH_THRESH
LOW_THRESH  = 10
HIGH_THRESH = 100
NUM_CLASSES = 3

CLASS_NAMES = {0: f"low (< {LOW_THRESH})",
               1: f"medium ({LOW_THRESH}–{HIGH_THRESH})",
               2: f"high (> {HIGH_THRESH})"}

# GMEL config
YEAR              = 2015
NUM_HIDDEN_LAYERS = 1
EMBEDDING_SIZE    = 128
MULTITASK_WEIGHTS = (0.5, 0.25, 0.25)
DEVICE            = "cpu"
device            = torch.device(DEVICE)

# How many hops for the k-hop subgraph (match GNN depth)
N_HOPS = NUM_HIDDEN_LAYERS + 1

# Target class for counterfactual — change to whatever class you want to reach
# Leave as None to auto-set to the next class above the current prediction
TARGET_CLASS = None

print("Config set.")

## 2. Load GMEL Data, Model, and Embeddings

In [ ]:
data_gmel  = utils.load_dataset(year=YEAR)
node_feats = data_gmel["node_feats"]          # (2168, 65) z-score normalised
ct_adj     = data_gmel["ct_adjacency_withweight"]
distm      = data_gmel["distm"]
num_nodes  = data_gmel["num_nodes"]

# Full baseline DGL graph
g_baseline = utils.build_graph_from_matrix(
    ct_adj, node_feats.astype(np.float32), device=DEVICE
).to(device)

# Load GMEL GNN checkpoint
ckpt_path = (
    f"./models/model_state_layers{NUM_HIDDEN_LAYERS}_"
    f"emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.pth"
)
gmel_model = MyModel(
    g=g_baseline, num_nodes=num_nodes,
    in_dim=node_feats.shape[1], h_dim=EMBEDDING_SIZE,
    num_hidden_layers=NUM_HIDDEN_LAYERS, device=DEVICE, reg_param=0
).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
gmel_model.load_state_dict(ckpt["state_dict"])
gmel_model.eval()

# Full-graph baseline embeddings (recomputed from checkpoint for consistency)
with torch.no_grad():
    src_emb_full = gmel_model(g_baseline).detach().cpu().numpy()   # (2168, 128)
    dst_emb_full = gmel_model.forward2(g_baseline).detach().cpu().numpy()

scaled_distm = distm / distm.max() * np.max([src_emb_full.max(), dst_emb_full.max()])

# Load GBRT
gbrt_path = (
    f"./models/gbrt_year{YEAR}_layers{NUM_HIDDEN_LAYERS}_"
    f"emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.pkl"
)
with open(gbrt_path, "rb") as f:
    gbrt = pickle.load(f)

print(f"Loaded. Nodes: {num_nodes}, feature dims: {node_feats.shape[1]}, embedding dims: {src_emb_full.shape[1]}")

## 3. Discretise Flow into Classes

For each node `i`, predict the flow `i → DST_NODE` using the full GMEL pipeline and assign a class.  
This becomes the node label `y` for the classification task COMBINEX will explain.

In [ ]:
def flow_to_class(flow):
    if flow < LOW_THRESH:
        return 0
    elif flow <= HIGH_THRESH:
        return 1
    else:
        return 2

# Predict flow from every node → DST_NODE using full-graph embeddings
all_src_idx  = np.arange(num_nodes)
feat_src     = src_emb_full[all_src_idx]                              # (2168, 128)
feat_dst     = np.tile(dst_emb_full[DST_NODE], (num_nodes, 1))        # (2168, 128)
feat_dist    = scaled_distm[all_src_idx, DST_NODE].reshape(-1, 1)     # (2168, 1)
X_to_dst     = np.concatenate([feat_src, feat_dst, feat_dist], axis=1)

flows_to_dst = gbrt.predict(X_to_dst)                                 # (2168,)
y_classes    = np.array([flow_to_class(f) for f in flows_to_dst], dtype=np.int64)

print(f"Flow range: {flows_to_dst.min():.1f} – {flows_to_dst.max():.1f}")
for c, name in CLASS_NAMES.items():
    print(f"  Class {c} [{name}]: {(y_classes == c).sum()} nodes")

src_class_orig = flow_to_class(flows_to_dst[SRC_NODE])
print(f"\nSRC_NODE {SRC_NODE} → DST_NODE {DST_NODE}:  "
      f"flow = {flows_to_dst[SRC_NODE]:.2f}  →  Class {src_class_orig} [{CLASS_NAMES[src_class_orig]}]")

## 4. Build PyG Data Object

COMBINEX uses PyTorch Geometric's `Data` format. We convert the GMEL graph here.  
`DataInfo` also requires `min_range`, `max_range`, and `discrete_mask` on the data object.

In [ ]:
# Convert DGL edge list → PyG edge_index
e_src, e_dst = g_baseline.edges()
edge_index = torch.stack([e_src.long(), e_dst.long()], dim=0)

x = torch.tensor(node_feats, dtype=torch.float32)
y = torch.tensor(y_classes,  dtype=torch.long)

# Extra fields required by DataInfo
min_range     = torch.tensor(node_feats.min(axis=0), dtype=torch.float32)
max_range     = torch.tensor(node_feats.max(axis=0), dtype=torch.float32)
discrete_mask = torch.zeros(node_feats.shape[1], dtype=torch.bool)  # all continuous

data_pyg = Data(
    x=x, edge_index=edge_index, y=y,
    min_range=min_range, max_range=max_range, discrete_mask=discrete_mask
)
data_pyg.train_mask = torch.ones(num_nodes,  dtype=torch.bool)
data_pyg.test_mask  = torch.zeros(num_nodes, dtype=torch.bool)

print(data_pyg)

## 5. Train Linear Probe

COMBINEX needs a **differentiable** oracle. GMEL's GBRT is not differentiable, so we train a logistic regression ("probe") on top of GMEL's source embeddings to predict flow class.  
The probe is then wrapped as a `torch.nn.Linear` so it can be used inside the oracle's `forward()`.

Input to probe: **source embedding only** (128-dim). Destination embedding and distance are fixed for a given DST_NODE, so the source embedding alone captures enough variance in flow class.

In [ ]:
from sklearn.linear_model import LogisticRegression

probe_sk = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
probe_sk.fit(src_emb_full, y_classes)

train_acc = probe_sk.score(src_emb_full, y_classes)
print(f"Probe train accuracy: {train_acc:.3f}")
print(f"Probe class predictions for SRC_NODE {SRC_NODE}: "
      f"class {probe_sk.predict(src_emb_full[[SRC_NODE]])[0]} "
      f"(ground truth class {src_class_orig})")

# Wrap as torch module so the oracle can call it in forward()
class LinearProbe(nn.Module):
    def __init__(self, sk_logreg, in_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes, bias=True)
        self.fc.weight.data = torch.tensor(sk_logreg.coef_,       dtype=torch.float32)
        self.fc.bias.data   = torch.tensor(sk_logreg.intercept_,  dtype=torch.float32)

    def forward(self, emb):
        return self.fc(emb)

probe = LinearProbe(probe_sk, EMBEDDING_SIZE, NUM_CLASSES)
probe.eval()
print("Probe converted to torch module.")

## 6. Define GMEL Oracle

The oracle wraps GMEL + the linear probe so it satisfies COMBINEX's interface:
- `forward(x, edge_index)` → `[num_nodes, num_classes]` log-softmax
- `get_embedding_repr(x, edge_index)` → `[num_nodes, embedding_dim]`

**Graph swapping:** GMEL's GAT layers store `self.g` at construction time. To run inference on an arbitrary subgraph, `swap_graph()` temporarily redirects the stored reference, runs forward, then restores it.

In [ ]:
def swap_graph(gmel, new_g):
    """Redirect all GAT layer graph references to new_g."""
    for layer in gmel.gat.layers:
        layer.g = new_g
    for layer in gmel.gat2.layers:
        layer.g = new_g

def make_dgl_from_pyg(x, edge_index, edge_weight_fill=0.5):
    """Build a minimal DGL graph from PyG tensors (x, edge_index)."""
    n = x.shape[0]
    src_np = edge_index[0].cpu().numpy()
    dst_np = edge_index[1].cpu().numpy()
    g = dgl.graph((src_np, dst_np), num_nodes=n)
    g.ndata['attr'] = x.float().cpu()
    g.edata['d']    = torch.full((g.num_edges(), 1), edge_weight_fill, dtype=torch.float32)
    in_deg  = g.in_degrees().float().numpy()
    norm    = np.zeros_like(in_deg)
    nonzero = in_deg > 0
    norm[nonzero] = 1.0 / in_deg[nonzero]
    g.ndata['norm'] = torch.from_numpy(norm).view(-1, 1)
    return g


class GMLFlowOracle(nn.Module):
    """
    Node-level classifier oracle for COMBINEX backed by GMEL.

    For each node in the (sub)graph, predicts the flow class for that node
    as origin toward DST_NODE using GMEL source embeddings + linear probe.
    """

    def __init__(self, num_features, num_classes, cfg,
                 gmel_model, probe, baseline_g):
        super().__init__()
        self.gmel       = gmel_model
        self.probe      = probe
        self.baseline_g = baseline_g

    def get_embedding_repr(self, x, edge_index):
        """Called by build_factual_graph. Returns node embeddings from GMEL GNN."""
        g = make_dgl_from_pyg(x, edge_index)
        swap_graph(self.gmel, g)
        with torch.no_grad():
            emb = self.gmel(g).cpu()
        swap_graph(self.gmel, self.baseline_g)   # restore
        return emb

    def forward(self, x, edge_index, edge_weights=None):
        """Returns log-softmax class probabilities — shape [num_nodes, num_classes]."""
        emb    = self.get_embedding_repr(x, edge_index)
        logits = self.probe(emb)
        return F.log_softmax(logits, dim=1)


print("Oracle class defined.")

## 7. Set Up COMBINEX Config and DataInfo

In [ ]:
# Build a minimal OmegaConf config matching the fields COMBINEX expects
cfg = OmegaConf.create({
    "device": DEVICE,
    "seed":   42,
    "task": {
        "name":        "Node",
        "folds_num":   10,
        "lr":          0.001,
        "dropout":     0.0,
        "weight_decay":0.001,
        "clip":        2.0,
        "hidden":      EMBEDDING_SIZE,
        "epochs":      450,
        "keep_ego":    True,
    },
    "model": {
        "name":           "gcn",
        "hidden_layers":  [EMBEDDING_SIZE],
    },
    "explainer": {
        "name":           "combined",
        "clip_grad_norm": 2.0,
    },
    "general": {"seed": 42},
    "verbose": False,
    "timeout": 500,
})

# DataInfo extracts num_features and num_classes from the PyG data object
# (it also deletes self.data, so pass a copy)
import copy
datainfo = DataInfo(cfg, copy.copy(data_pyg))

print(f"DataInfo — num_features: {datainfo.num_features}, num_classes: {datainfo.num_classes}")

In [ ]:
# Instantiate oracle
oracle = GMLFlowOracle(
    num_features=datainfo.num_features,
    num_classes=datainfo.num_classes,
    cfg=cfg,
    gmel_model=gmel_model,
    probe=probe,
    baseline_g=g_baseline,
).to(device)
oracle.eval()

# Get full-graph predictions
with torch.no_grad():
    out_full = oracle(data_pyg.x, data_pyg.edge_index)
    predicted_labels = torch.argmax(out_full, dim=1)

src_predicted_class = int(predicted_labels[SRC_NODE].item())

# Decide target class
if TARGET_CLASS is None:
    target_class = (src_predicted_class + 1) % NUM_CLASSES
else:
    target_class = TARGET_CLASS

target_labels = torch.full((num_nodes,), target_class, dtype=torch.long)

print(f"Oracle predicts SRC_NODE {SRC_NODE} as class {src_predicted_class} [{CLASS_NAMES[src_predicted_class]}]")
print(f"Counterfactual target class: {target_class} [{CLASS_NAMES[target_class]}]")

## 8. Build Factual Graph and Run COMBINEX

In [ ]:
# Extract k-hop subgraph around SRC_NODE
factual_graph = build_factual_graph(
    mask_index=SRC_NODE,
    data=data_pyg,
    n_hops=N_HOPS,
    oracle=oracle,
    predicted_labels=predicted_labels,
    target_labels=target_labels,
    device=DEVICE,
)

assert not check_graphs(factual_graph.edge_index), "Invalid factual graph."

print("Factual graph:", factual_graph)
print(f"Center node local index: {factual_graph.new_idx}")
print(f"Subgraph size: {factual_graph.num_nodes} nodes, {factual_graph.edge_index.shape[1]} edges")

In [ ]:
# Run COMBINEX explainer
ExplainerCls  = get_node_explainer("combined")
explainer     = ExplainerCls(cfg, datainfo)

counterfactual = explainer.explain(graph=factual_graph, oracle=oracle)
print("Counterfactual graph:", counterfactual)

# Verify class flipped
with torch.no_grad():
    cf_logits = oracle(counterfactual.x, counterfactual.edge_index)
    cf_labels = torch.argmax(cf_logits, dim=1)

center_local = int(factual_graph.new_idx)
orig_class_local = int(predicted_labels[SRC_NODE].item())
cf_class_local   = int(cf_labels[center_local].item())

print(f"\nCenter node (local {center_local}) — original class: {orig_class_local} [{CLASS_NAMES[orig_class_local]}]")
print(f"Center node (local {center_local}) — CF class:       {cf_class_local} [{CLASS_NAMES[cf_class_local]}]")
print("Class flipped!" if cf_class_local != orig_class_local else "Class NOT flipped — COMBINEX did not find a CF.")

## 9. Analyse Counterfactual Changes

Which node features changed, and by how much?  
The changes are in **z-score normalised units** (same as the GMEL input space).

In [ ]:
from collections import deque

MAPPING_PATH = "../data/CensusTract2010/mapping_NodeID2BoroCT2010.csv"
FEAT_PATH    = "../data/PLUTO/census_tract_attributes_from_pluto2015.csv"

mapping_df   = pd.read_csv(MAPPING_PATH)
node2ct      = mapping_df.set_index("node_id")["BoroCT2010"].to_dict()
feat_cols    = pd.read_csv(FEAT_PATH, nrows=0).columns.tolist()
feat_cols.remove("BoroCT2010")

# Recover raw mean/std for back-transforming delta to raw LQ units
raw_pluto    = pd.read_csv(FEAT_PATH).set_index("BoroCT2010")[feat_cols]
feat_mean_raw = raw_pluto.mean().values
feat_std_raw  = raw_pluto.std().values

def hop_distances(edge_index, center, num_nodes):
    adj = [[] for _ in range(num_nodes)]
    for u, v in edge_index.t().tolist():
        adj[u].append(v); adj[v].append(u)
    dist = [-1] * num_nodes
    dist[center] = 0
    q = deque([center])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

def print_feature_changes(factual_g, cf_g, local_nodes, feat_cols, feat_mean_raw, feat_std_raw, topk=8):
    x0 = factual_g.x.detach().cpu()
    x1 = cf_g.x.detach().cpu()
    node_dict_inv = {v: k for k, v in factual_g.node_dict.items()}  # local → global

    for n in local_nodes:
        diff    = x1[n] - x0[n]
        abs_diff = diff.abs()
        changed  = (abs_diff > 1e-6).sum().item()
        if changed == 0:
            continue

        global_idx = node_dict_inv.get(n, "?")
        ct_code    = node2ct.get(global_idx, "?")
        print(f"  Node local={n}  global={global_idx}  BoroCT2010={ct_code}  ({changed} features changed)")

        topk_idx = torch.topk(abs_diff, k=min(topk, len(feat_cols))).indices.tolist()
        for j in topk_idx:
            if abs_diff[j].item() < 1e-6:
                continue
            feat   = feat_cols[j]
            before = float(x0[n, j].item())
            after  = float(x1[n, j].item())
            delta_norm = float(diff[j].item())
            # Convert to raw LQ delta
            delta_raw  = delta_norm * feat_std_raw[j]
            raw_before = feat_mean_raw[j] + before * feat_std_raw[j]
            raw_after  = feat_mean_raw[j] + after  * feat_std_raw[j]
            print(f"    {feat:<35s}  "
                  f"{raw_before:>8.3f} → {raw_after:>8.3f}  (Δ raw LQ = {delta_raw:+.4f}, Δ norm = {delta_norm:+.4f} σ)")

print("Analysis helpers defined.")

In [ ]:
dist = hop_distances(factual_graph.edge_index, center_local, factual_graph.num_nodes)
hop0 = [i for i, d in enumerate(dist) if d == 0]
hop1 = [i for i, d in enumerate(dist) if d == 1]
hop2 = [i for i, d in enumerate(dist) if d == 2]

print(f"Subgraph: {factual_graph.num_nodes} nodes — center (hop-0): {hop0}")
print(f"  Hop-1 neighbors: {len(hop1)} nodes")
print(f"  Hop-2 neighbors: {len(hop2)} nodes")

print(f"\n{'='*60}")
print(f"  CF: class {orig_class_local} [{CLASS_NAMES[orig_class_local]}]  →  "
      f"class {cf_class_local} [{CLASS_NAMES[cf_class_local]}]")
print(f"  SRC_NODE {SRC_NODE} (BoroCT2010 {node2ct[SRC_NODE]})  →  DST_NODE {DST_NODE}")
print(f"{'='*60}")

print("\n── HOP 0 (centre — SRC_NODE) ──────────────────────────────")
print_feature_changes(factual_graph, counterfactual, hop0,
                      feat_cols, feat_mean_raw, feat_std_raw, topk=10)

print("\n── HOP 1 (direct neighbours) ───────────────────────────────")
print_feature_changes(factual_graph, counterfactual, hop1,
                      feat_cols, feat_mean_raw, feat_std_raw, topk=5)

print("\n── HOP 2 (2-hop neighbours) ────────────────────────────────")
print_feature_changes(factual_graph, counterfactual, hop2,
                      feat_cols, feat_mean_raw, feat_std_raw, topk=3)

## 10. Verify With Full GMEL Pipeline

Take the CF node-feature changes at hop-0 (SRC_NODE), apply them to the full graph,
and check whether the actual GMEL → GBRT predicted flow also shifts class.

In [ ]:
# Extract CF feature vector for SRC_NODE (local index = center_local)
cf_feats_center = counterfactual.x[center_local].detach().cpu().numpy()  # (65,)

# Apply to full node feature matrix
node_feats_cf = node_feats.copy()
node_feats_cf[SRC_NODE] = cf_feats_center

# Rebuild full graph with CF features
g_cf = utils.build_graph_from_matrix(
    ct_adj, node_feats_cf.astype(np.float32), device=DEVICE
).to(device)

# Run GMEL GNN on full CF graph
swap_graph(gmel_model, g_cf)
with torch.no_grad():
    src_emb_cf = gmel_model(g_cf).detach().cpu().numpy()
    dst_emb_cf = gmel_model.forward2(g_cf).detach().cpu().numpy()
swap_graph(gmel_model, g_baseline)   # restore

# Predict flow for SRC_NODE → DST_NODE
scaled_distm_cf = distm / distm.max() * np.max([src_emb_cf.max(), dst_emb_cf.max()])
pair = np.array([[SRC_NODE, DST_NODE]], dtype=int)

feat_orig = np.concatenate([
    src_emb_full[[SRC_NODE]], dst_emb_full[[DST_NODE]],
    scaled_distm[[SRC_NODE], DST_NODE].reshape(1, 1)
], axis=1)
feat_cf = np.concatenate([
    src_emb_cf[[SRC_NODE]], dst_emb_cf[[DST_NODE]],
    scaled_distm_cf[[SRC_NODE], DST_NODE].reshape(1, 1)
], axis=1)

flow_orig = gbrt.predict(feat_orig)[0]
flow_cf   = gbrt.predict(feat_cf)[0]

class_orig = flow_to_class(flow_orig)
class_cf   = flow_to_class(flow_cf)

print(f"{'='*55}")
print(f"  Full GMEL pipeline verification")
print(f"  SRC {SRC_NODE} → DST {DST_NODE}")
print(f"{'='*55}")
print(f"  Original flow : {flow_orig:.2f}  → class {class_orig} [{CLASS_NAMES[class_orig]}]")
print(f"  CF flow       : {flow_cf:.2f}  → class {class_cf} [{CLASS_NAMES[class_cf]}]")
print(f"  Difference    : {flow_cf - flow_orig:+.2f}")
print(f"{'='*55}")
if class_cf != class_orig:
    print("  ✓ Class flipped in full GMEL pipeline too!")
else:
    print("  ✗ Class did NOT flip in full GMEL pipeline (oracle approximation only).")